# 更多有趣的词语向量
## 单词的数字表示

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# 安装在part2中训练好的模型
from gensim.models import Word2Vec
model = Word2Vec.load("300features_40minwords_10context")
type(model.wv.vectors)
model.wv.vectors.shape
model.wv["flower"]

## 从文字到段落，尝试1：向量平均法
Word2Vec 只能得到单个单词的向量，但我们任务处理的是整篇评论、段落（一长串单词）。  
一篇文本有 N 个单词，N 是不固定的，有的句子 3 个词，有的 300 个词。普通机器学习模型（SVM、逻辑回归、随机森林）只能接收固定长度的数字数组作为输入，不能直接接收长短不一的单词列表。  
平均向量，就是把任意长度文本，压缩成一段固定长度的数字向量。  
举例子：词向量维度 300  
一句话 5 个单词 → 平均后输出：1×300 的向量  
一段话 200 个单词 → 平均后输出：1×300 的向量  
不管原文多长，输出永远是 300 个数字，可以直接丢给分类模型。  

In [ ]:
import numpy as np  

def makeFeatureVec(words, model, num_features):
    
    # 初始化全 0 向量，用来累加所有单词向量。
    featureVec = np.zeros((num_features,),dtype="float32")
    
    #统计这篇文本中，出现在词表中的有效单词数量。
    nwords = 0.
    
    # model.index2word 是模型全部词汇列表；转成set集合，查询word in set速度很快。
    index2word_set = set(model.index2word)
    
    for word in words:
        if word in index2word_set: 
            nwords = nwords + 1.
            featureVec = np.add(featureVec,model[word])
    
    featureVec = np.divide(featureVec,nwords)
    return featureVec


def getAvgFeatureVecs(reviews, model, num_features):
    
    counter = 0.
    
    reviewFeatureVecs = np.zeros((len(reviews),num_features),dtype="float32")

    for review in reviews:
       if counter%1000. == 0.:
           print "Review %d of %d" % (counter, len(reviews))
     
       reviewFeatureVecs[counter] = makeFeatureVec(review, model, \
           num_features)
    
       counter = counter + 1.
    return reviewFeatureVecs

现在，我们可以调用这些函数，为每个段落生成平均向量。

In [ ]:
clean_train_reviews = []
for review in train["review"]:
    clean_train_reviews.append( review_to_wordlist( review, \
        remove_stopwords=True ))

trainDataVecs = getAvgFeatureVecs( clean_train_reviews, model, num_features )

print ("Creating average feature vecs for test reviews")
for review in test["review"]:
    clean_test_reviews.append( review_to_wordlist( review, \
        remove_stopwords=True ))

testDataVecs = getAvgFeatureVecs( clean_test_reviews, model, num_features )

接下来，使用平均段落向量来训练一个随机森林。

In [ ]:
from sklearn.ensemble import RandomForestClassifier
forest = RandomForestClassifier( n_estimators = 100 )

print ("Fitting a random forest to labeled training data...")
forest = forest.fit( trainDataVecs, train["sentiment"] )

result = forest.predict( testDataVecs )

output = pd.DataFrame( data={"id":test["id"], "sentiment":result} )
output.to_csv( "Word2Vec_AverageVectors.csv", index=False, quoting=3 )

## 从文字到段落，尝试2：聚类  
Word2Vec 创建语义相关的词群，因此另一种可能的方法是利用词群内词的相似性。以这种方式分组向量称为“向量量化”。为此，我们首先需要找到词汇聚的中心，这可以通过使用如K-均值（K-Means）这样的聚类算法来实现。  

在K均值中，我们需要设置的参数是“K”，即簇的数量。我们应该如何决定创建多少集群？反复试验表明，平均每个簇只有大约5个单词的小集群，比单词众多的大集群效果更好。聚类代码如下所示。我们使用 scikit-learn 来执行我们的 K-Means。

In [ ]:
from sklearn.cluster import KMeans

word_vectors = model.wv.vectors
num_clusters = word_vectors.shape[0] / 5

kmeans_clustering = KMeans( n_clusters = num_clusters )

idx = kmeans_clustering.fit_predict( word_vectors )

idx是一维数组，长度等于词汇总数。idx[i]代表第 i 个单词被分到几号簇

将这些内容压缩到一个词典中，具体如下：idxmodel.index2word

In [ ]:
word_centroid_map = dict(zip( model.wv.index2word, idx ))

zip(A,B)：  
把两个等长序列一一配对：   
(model.index2word[0], idx[0])，(model.index2word[1], idx[1])……   
也就是：(单词字符串, 簇编号) 成对。  
dict(...)   
把配对结果转为 python 字典。   
最终：word_centroid_map["good"] → 返回该单词所属的簇号（整数）。     
后面处理一篇影评文本的时候：   
遍历句子里每一个单词，查这个字典拿到簇编号；   
统计各个簇编号出现多少次，就得到「聚类词袋特征」。   

这里有一个循环，可以打印出群组0到9的单词：

In [ ]:
for cluster in range(0,10):
    
    print ("\nCluster %d" % cluster)
    
    words = []
    for i in range(0,len(word_centroid_map.values())):
        if( word_centroid_map.values()[i] == cluster ):
            words.append(word_centroid_map.keys()[i])
    print (words)

现在我们为每个词都有一个簇（或称“重心”）分配，并且可以定义一个函数将评论转换为重心袋。这与Bag of Words的工作方式类似，但使用语义相关的簇而非单个单词：  
原来的词袋 BoW：统计句子里每个单词出现多少次，词典大小上万维，稀疏向量。  
Bag‑of‑Centroids（簇词袋）：不用单词，改用簇编号  
把一句影评切分成单词列表  
遍历句子里每一个词：  
如果词在词表中，取出它对应的簇编号  
统计：每个簇编号在这句话里一共出现多少次  
最终得到这条句子的特征：长度 = num_clusters 的向量，向量每一位代表该簇在本句出现频次  

In [ ]:
def create_bag_of_centroids( wordlist, word_centroid_map ):
    #簇从0开始算，所以要+1
    num_centroids = max( word_centroid_map.values() ) + 1

    bag_of_centroids = np.zeros( num_centroids, dtype="float32" )

    for word in wordlist:
        if word in word_centroid_map:
            index = word_centroid_map[word]
            bag_of_centroids[index] += 1
            
    return bag_of_centroids

In [ ]:
train_centroids = np.zeros( (train["review"].size, num_clusters), \
    dtype="float32" )

counter = 0
for review in clean_train_reviews:
    train_centroids[counter] = create_bag_of_centroids( review, \
        word_centroid_map )
    counter += 1

test_centroids = np.zeros(( test["review"].size, num_clusters), \
    dtype="float32" )

counter = 0
for review in clean_test_reviews:
    test_centroids[counter] = create_bag_of_centroids( review, \
        word_centroid_map )
    counter += 1

In [ ]:
forest = RandomForestClassifier(n_estimators = 100)

print ("用带标签的训练数据拟合随机森林模型")
forest = forest.fit(train_centroids,train["sentiment"])
result = forest.predict(test_centroids)

output = pd.DataFrame(data={"id":test["id"], "sentiment":result})
output.to_csv( "BagOfCentroids.csv", index=False, quoting=3 )